# LesioScan — Single Image Prediction + Grad-CAM

**Purpose:** Test the trained **ShuffleNet V2 x1.0** directly from your Google Drive checkpoint. Upload an image, inspect six class scores, and visualize model attention via **Grad-CAM**.

**Research demo only:** These outputs cannot diagnose skin disease. Grad-CAM visualizes model sensitivity, not medical evidence or a verified lesion boundary. No retraining or test-set evaluation occurs here.

## 1. Setup — Drive, GitHub, and device

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util

# Run this notebook on Google Colab. CPU is sufficient for one image.
from google.colab import drive
drive.mount('/content/drive')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/LesioScan-AI')
TRAIN_OUT = ROOT / 'experiments/training/shufflenet_v2_baseline'
MODEL_DIR = TRAIN_OUT / 'fine_tuning'
EVAL_CKPT = MODEL_DIR / 'best_finetuning_eval.pt'
FULL_CKPT = MODEL_DIR / 'best_finetuning.pt'
REPO = Path('/content/lesioscan-ml')
REPO_URL = 'https://github.com/Liex2810/lesioscan-ml.git'

if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))
from lesioscan.training import make_model
from lesioscan.preprocessing import build_transforms, CLASSES

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Classes:', list(CLASSES))
print('Eval checkpoint exists:', EVAL_CKPT.is_file())
print('Full checkpoint exists:', FULL_CKPT.is_file())


## 2. Load best checkpoint — no retraining

In [ ]:
# Prefer sanitized evaluation checkpoint: safe to read with weights_only=True.
# If only the older full training checkpoint exists, set TRUST_OWN_CHECKPOINT=True
# ONLY if you created and control that file yourself. weights_only=False can execute
# code embedded in an untrusted pickle.
TRUST_OWN_CHECKPOINT = False

if EVAL_CKPT.is_file():
    checkpoint = torch.load(EVAL_CKPT, map_location='cpu', weights_only=True)
    source_ckpt = EVAL_CKPT
elif FULL_CKPT.is_file() and TRUST_OWN_CHECKPOINT:
    original = torch.load(FULL_CKPT, map_location='cpu', weights_only=False)
    checkpoint = {
        'model': original['model'],
        'stage': original['stage'],
        'epoch': int(original['epoch']),
        'classes': list(original['classes']),
        'image_size': int(original.get('image_size', 224)),
        'model_state_dict': original['model_state_dict'],
    }
    torch.save(checkpoint, EVAL_CKPT)
    source_ckpt = EVAL_CKPT
else:
    raise FileNotFoundError(
        f'Missing safe evaluation checkpoint: {EVAL_CKPT}. '
        'If you only have your trusted original best_finetuning.pt, '
        'set TRUST_OWN_CHECKPOINT=True and rerun this cell.'
    )

assert checkpoint['model'] == 'shufflenet_v2_x1_0', 'Unexpected model architecture'
assert list(checkpoint['classes']) == list(CLASSES), 'Class order mismatch!'
image_size = int(checkpoint.get('image_size', 224))
model = make_model(pretrained=False)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model = model.to(DEVICE).eval()
_, eval_transform = build_transforms(image_size)

print('✅ Loaded:', source_ckpt)
print('Stage:', checkpoint.get('stage'))
print('Best epoch:', checkpoint.get('epoch'))
print('Image size:', image_size)


## 3. Upload one JPG/PNG image

In [ ]:
from google.colab import files
from io import BytesIO

uploaded = files.upload()
if not uploaded:
    raise ValueError('Choose at least one JPG or PNG image.')

filename, image_bytes = next(iter(uploaded.items()))
with Image.open(BytesIO(image_bytes)) as f:
    original_image = ImageOps.exif_transpose(f).convert('RGB')

print('Filename:', filename)
print('Original dimensions:', original_image.size)
display(original_image.resize((350, 350)))


## 4. Predictions — all six classes

In [ ]:
input_tensor = eval_transform(original_image).unsqueeze(0).to(DEVICE)
assert tuple(input_tensor.shape) == (1, 3, image_size, image_size)

with torch.inference_mode():
    logits = model(input_tensor)
    scores = logits.softmax(dim=1)[0].cpu().numpy()

pred_id = int(np.argmax(scores))
pred_class = str(CLASSES[pred_id])
score_table = pd.DataFrame({'class': list(CLASSES), 'model_score': scores, 'score_percent': scores * 100})
score_table = score_table.sort_values('model_score', ascending=False).reset_index(drop=True)

print(f'Predicted class: {pred_class} | Model score: {scores[pred_id]*100:.2f}%')
display(score_table.round(3))
print('⚠️ Softmax scores are NOT calibrated clinical probabilities.')


## 5. Grad-CAM — attention heatmap

Grad-CAM uses gradients with respect to the final convolutional feature maps of ShuffleNet V2 (`model.conv5`). It highlights image regions contributing to the **selected class score**; it does **not** segment a lesion or verify a diagnosis. The overlay uses the same deterministic evaluation transformation as the model input.

In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, net, target_layer):
        self.net = net
        self.activations = None
        self.gradients = None
        self.handles = [
            target_layer.register_forward_hook(self._save_activation),
            target_layer.register_full_backward_hook(self._save_gradient),
        ]

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, tensor, class_index):
        # Forward/backward must use gradients (NOT inference_mode or no_grad).
        self.net.eval()
        self.net.zero_grad(set_to_none=True)
        with torch.enable_grad():
            output = self.net(tensor.detach())
            output[0, class_index].backward()
        if self.gradients is None or self.activations is None:
            raise RuntimeError('Grad-CAM hooks did not capture features/gradients.')
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True).relu()
        cam = F.interpolate(cam, size=tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        self.net.zero_grad(set_to_none=True)
        return cam

    def close(self):
        for h in self.handles:
            h.remove()

# Choose what to explain: default = top prediction.
# To inspect a specific class, replace e.g. 'MEL'.
TARGET_CLASS = pred_class
assert TARGET_CLASS in CLASSES
cam_explainer = GradCAM(model, model.conv5)
try:
    cam = cam_explainer.generate(input_tensor, list(CLASSES).index(TARGET_CLASS))
finally:
    cam_explainer.close()

# Denormalize model input, so the overlay matches the pixels actually seen by model.
# ImageNet statistics, consistent with standard pretrained ShuffleNet transforms.
# Prefer display of the exact evaluation-transformed tensor if preprocessing differs.
mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3,1,1)
pixel_tensor = input_tensor[0].detach().cpu()
seen_image = (pixel_tensor * std + mean).clamp(0, 1).permute(1,2,0).numpy()

heat_rgb = plt.get_cmap('jet')(cam)[..., :3]
overlay = np.clip(0.55 * seen_image + 0.45 * heat_rgb, 0, 1)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(original_image); ax[0].set_title('Original image')
ax[1].imshow(cam, cmap='jet', vmin=0, vmax=1); ax[1].set_title(f'Grad-CAM heatmap — {TARGET_CLASS}')
ax[2].imshow(overlay); ax[2].set_title(f'Grad-CAM overlay — {TARGET_CLASS}')
for a in ax: a.axis('off')
fig.suptitle(f'LesioScan | Predicted: {pred_class} ({scores[pred_id]*100:.1f}% model score)')
fig.tight_layout()
plt.show()


## 6. Optional — plot class scores and save visualizations

In [ ]:
from datetime import datetime
SAVE_TO_DRIVE = False  # Change to True to save outputs in your Drive.

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(score_table['class'], score_table['score_percent'])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel('Uncalibrated softmax score (%)')
ax.set_title('LesioScan — six-class scores')
for i, v in enumerate(score_table['score_percent']):
    ax.text(min(float(v)+1, 95), i, f'{v:.1f}%', va='center')
fig.tight_layout()
plt.show()

if SAVE_TO_DRIVE:
    save_dir = ROOT / 'experiments' / 'demo_inference' / datetime.now().strftime('%Y%m%d_%H%M%S')
    save_dir.mkdir(parents=True, exist_ok=True)
    score_table.to_csv(save_dir / 'class_scores.csv', index=False)
    Image.fromarray((np.clip(overlay, 0, 1)*255).astype(np.uint8)).save(save_dir / 'gradcam_overlay.png')
    Image.fromarray((np.clip(heat_rgb, 0, 1)*255).astype(np.uint8)).save(save_dir / 'gradcam_heatmap.png')
    print('Saved:', save_dir)
else:
    print('Not saved to Drive (SAVE_TO_DRIVE=False).')


## Interpretation and limitations

- **Predicted class:** the class with the highest model score, **not a diagnosis**.
- **Heatmap:** relative influence of regions on one class score; it may highlight hair, edges, background, or artifacts rather than the lesion.
- **Unseen photos:** may differ substantially from PAD-UFES-20. Never interpret model output as a medical recommendation.
- **Reproducibility:** the checkpoint, class order, and `build_transforms()` must match training. To test another image, rerun sections 3–6.